In [17]:
import pandas as pd
import requests
import time

In [18]:
# 1. 환경 설정
KAKAO_API_KEY = "620f3722c76e8e90e618f775b44927a3"
SUBWAY_PATH = '../data/subway_data.csv'
SHOP_PATH = '../data/seoul_shop_data.csv'

In [28]:
# 2. 데이터 불러오기 및 정제
def load_data(path, is_shop=False):
    for enc in ['utf-8-sig', 'cp949', 'utf-8']:
        try:
            if is_shop:
                # 상권 데이터는 필요한 컬럼만 선택 로드
                cols = ['상호명', '상권업종소분류명', '행정동명', '경도', '위도']
                return pd.read_csv(path, usecols=cols, encoding=enc)
            else:
                # 지하철 데이터 전체 로드
                return pd.read_csv(path, encoding=enc)
        except:
            continue
    return None

# 데이터 불러오기
subway = load_data(SUBWAY_PATH)
shop = load_data(SHOP_PATH, is_shop=True)

# 컬럼명에 있을 수 있는 공백 제거
subway.columns = subway.columns.str.strip()
shop.columns = shop.columns.str.strip()

# 지하철 좌표 컬럼 유연하게 처리
x_col = '환승역X좌표' if '환승역X좌표' in subway.columns else 'X좌표'
y_col = '환승역Y좌표' if '환승역Y좌표' in subway.columns else 'Y좌표'

# 상권 데이터 좌표명 통일
shop = shop.rename(columns={'경도': 'X좌표', '위도': 'Y좌표'})

In [29]:
# 3. 카카오 API 행정동 추출 함수 (보정 로직 추가)
def get_administrative_dong(x, y):
    if pd.isna(x) or pd.isna(y) or x == 0: return None
    
    url = "https://dapi.kakao.com/v2/local/geo/coord2regioncode.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    
    try:
        res = requests.get(url, headers=headers, params={"x": x, "y": y}, timeout=5)
        if res.status_code == 200:
            docs = res.json().get('documents', [])
            for region in docs:
                # 'H'는 행정동, 'B'는 법정동입니다. 행정동을 우선합니다.
                if region.get('region_type') == 'H':
                    return region.get('region_3depth_name') # "역삼1동" 형태
        return None
    except:
        return None

In [30]:
# 4. 지하철 데이터 행정동 매핑 (3~4분 소요)
subway['행정동명'] = subway.apply(lambda row: get_administrative_dong(row[x_col], row[y_col]), axis=1)

# 지하철역 개수 집계 및 텍스트 정제
subway['행정동명'] = subway['행정동명'].str.strip()
subway_counts = subway.groupby('행정동명').size().reset_index(name='지하철역')

In [31]:
# 5. 상권 데이터 업종별 집계
infra_counts = shop.groupby(['행정동명', '상권업종소분류명']).size().unstack(fill_value=0)

mapping = {
    # 1. 식생활 및 휴식
    '카페': '카페',
    '커피전문점/카페': '카페',
    '치킨': '음식점',
    '중국집': '음식점',
    '백반/한정식': '음식점',
    '요리 주점': '음식점',
    '횟집': '음식점',
    '일식 회/초밥': '음식점',
    '패스트푸드': '음식점',
    '서양식 음식점': '음식점',
    '국수/칼국수': '음식점',

    # 2. 생활 편의 (편의점/마트)
    '편의점': '편의점',
    '슈퍼마켓': '마트',
    '식료품 소매업': '마트',
    '종합 소매점': '마트',

    # 3. 의료 인프라 (진료과별 통합)
    '약국': '약국',
    '치과의원': '병원',
    '피부/비뇨기과 의원': '병원',
    '내과 의원': '병원',
    '소아청소년과 의원': '병원',
    '이비인후과 의원': '병원',
    '산부인과 의원': '병원',
    '일반 의원': '병원',
    '한의원': '병원',

    # 4. 문화 및 여가시설
    '영화관': '영화관',
    '노래방': '여가시설',
    'PC방': '여가시설',
    '당구장': '여가시설',

    # 5. 운동 및 건강
    '헬스장': '운동시설',
    '요가/필라테스': '운동시설',
    '실내 골프 연습장': '운동시설',
    '수영장': '운동시설',

    # 6. 교육 인프라
    '입시·교과학원': '학원',
    '외국어 학원': '학원',
    '음악학원': '학원',
    '미술학원': '학원',
    '태권도장': '학원',
    '독서실': '공부방',

    # 7. 생활 서비스
    '미용실': '미용실',
    '세탁소': '세탁소',
    '자동차 수리업': '정비소'
}

infra_final = pd.DataFrame(index=infra_counts.index)
for orig, target in mapping.items():
    if orig in infra_counts.columns:
        if target not in infra_final.columns: infra_final[target] = infra_counts[orig]
        else: infra_final[target] += infra_counts[orig]

infra_final = infra_final.reset_index()
infra_final['행정동명'] = infra_final['행정동명'].str.strip()

In [32]:
# 6. 데이터 병합
final_master = pd.merge(infra_final, subway_counts, on='행정동명', how='left').fillna(0)

In [34]:
# 7. 결과 저장
final_master.to_csv('../data/dataset.csv', index=False, encoding='utf-8-sig')